In [1037]:
import pandas as pd
from pathlib import Path

# Annotation Review
Reviewed by: Quinn Hinde-Schuster

## Purpose

This notebook documents an independent review of the Contract annotations for the selected categories. The goal is to evaluate whether the annotated Contract language is clear, consistent, and appropriate for each category without relying on the existing annotation labels during the initial review.

# Set Up

## 1. Data Used for the Independent Review

The independent review uses the training-approved Contract data and the frozen
train/validation split. Only the training portion is used for this review.

The review evidence will be drawn from the selected categories and their
associated Contract annotation spans. Existing annotation labels will be kept
separate from the independent decision so that they can be compared after the
independent review is completed.

The evidence used in the review will be recorded by Contract ID,
context group ID, category, and annotation/span ID.


In [1038]:
DATA_DIR = Path("split_data")
SPLIT_DIR = Path("frozen-split_data")

contracts = pd.read_parquet(DATA_DIR / "contracts.parquet")
documents = pd.read_parquet(DATA_DIR / "documents.parquet")
categories = pd.read_parquet(DATA_DIR / "categories.parquet")
annotation_sets = pd.read_parquet(DATA_DIR / "annotation_sets.parquet")
spans = pd.read_parquet(DATA_DIR / "spans.parquet")
split_map = pd.read_parquet(SPLIT_DIR / "frozen-split.parquet")

print(f"Contracts: {len(contracts)}")
print(f"Documents: {len(documents)}")
print(f"Categories: {len(categories)}")
print(f"Annotation sets: {len(annotation_sets)}")
print(f"Spans: {len(spans)}")
print(f"Split assignments: {len(split_map)}")

Contracts: 408
Documents: 408
Categories: 41
Annotation sets: 16728
Spans: 11180
Split assignments: 407


Verify the split

In [1039]:
documents = documents.merge(
    split_map,
    on="context_group_id",
    how="left"
)

unmatched = documents["split"].isna().sum()

print(f"Documents with no split assignment: {unmatched}")
print(documents["split"].value_counts())

Documents with no split assignment: 0
split
train         325
validation     83
Name: count, dtype: int64


## 2. Selected Categories

The independent review covers the 10 categories selected for the challenge.
These categories are:

1. Governing Law
2. Renewal Term
3. Revenue/Profit Sharing
4. Cap On Liability
5. Uncapped Liability
6. Termination For Convenience
7. Anti-Assignment
8. Audit Rights
9. License Grant
10. Exclusivity

I will review evidence associated with these categories independently before
comparing my decisions with the existing annotations done by Amie and Aishat in task 7.

In [1040]:
final_cat_names = [
    "Governing Law",
    "Renewal Term",
    "Revenue/Profit Sharing",
    "Cap On Liability",
    "Uncapped Liability",
    "Termination For Convenience",
    "Anti-Assignment",
    "Audit Rights",
    "License Grant",
    "Exclusivity",
]

final_categories = categories[
    categories["category_name"].isin(final_cat_names)
].copy()

print(final_categories[["category_id", "category_name"]].to_string(index=False))

                category_id               category_name
               renewal_term                Renewal Term
              governing_law               Governing Law
                exclusivity                 Exclusivity
termination_for_convenience Termination For Convenience
            anti_assignment             Anti-Assignment
     revenue_profit_sharing      Revenue/Profit Sharing
              license_grant               License Grant
               audit_rights                Audit Rights
         uncapped_liability          Uncapped Liability
           cap_on_liability            Cap On Liability


## 3. Constructing the Independent Review Evidence

For the independent review, I'm creating a review set containing Contract
excerpts associated with the 10 selected categories.

The existing annotation decision are not shown during the initial review.
For each excerpt, I'm going to independently determine whether the language should
count for the category and record my reasoning for that decision.

After the independent decisions are recorded, they will be compared with the
existing annotations to identify agreements, disagreements, and unclear cases.

In [1041]:
# Get training documents/contracts
train_documents = documents[documents["split"] == "train"].copy()
train_contract_ids = set(train_documents["contract_id"])

train_contracts = contracts[
    contracts["contract_id"].isin(train_contract_ids)
].copy()

train_annotation_sets = annotation_sets[
    annotation_sets["contract_id"].isin(train_contract_ids)
].copy()

train_spans = spans[
    spans["annotation_set_id"].isin(
        train_annotation_sets["annotation_set_id"]
    )
].copy()

print(f"Training contracts: {len(train_contracts)}")
print(f"Training annotation sets: {len(train_annotation_sets)}")
print(f"Training spans: {len(train_spans)}")

Training contracts: 325
Training annotation sets: 13325
Training spans: 8964


## 4. Understanding the Annotation Evidence

Here I am inspecting how annotation sets
and spans are represented in the dataset. This is necessary to distinguish
the Contract text being reviewed from the existing annotation decision.

The existing labels will be kept separate from the evidence used for the
initial independent review.

In [1042]:
print("Annotation set columns:")
print(annotation_sets.columns.tolist())

print("\nSpan columns:")
print(spans.columns.tolist())

print("\nExample annotation set:")
display(annotation_sets.head())

print("\nExample span:")
display(spans.head())

Annotation set columns:
['annotation_set_id', 'contract_id', 'category_id', 'is_impossible']

Span columns:
['span_id', 'annotation_set_id', 'source_qa_id', 'answer_text', 'answer_start', 'answer_end']

Example annotation set:


,annotation_set_id,contract_id,category_id,is_impossible
0,contract_0001__affiliate_license_licensee,contract_0001,affiliate_license_licensee,True
1,contract_0001__affiliate_license_licensor,contract_0001,affiliate_license_licensor,True
2,contract_0001__agreement_date,contract_0001,agreement_date,False
3,contract_0001__anti_assignment,contract_0001,anti_assignment,False
4,contract_0001__audit_rights,contract_0001,audit_rights,True



Example span:


,span_id,annotation_set_id,source_qa_id,answer_text,answer_start,answer_end
0,contract_0001__document_name__span_000,contract_0001__document_name,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,DISTRIBUTOR AGREEMENT,44,65
1,contract_0001__parties__span_000,contract_0001__parties,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Distributor,244,255
2,contract_0001__parties__span_001,contract_0001__parties,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Electric City of Illinois L.L.C.,49574,49606
3,contract_0001__parties__span_002,contract_0001__parties,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Electric City of Illinois LLC,212,241
4,contract_0001__parties__span_003,contract_0001__parties,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Company,197,204


Here we see that each annotation set connects a Contract to a specific
category, while the spans contain the specific pieces of Contract text
associated with that category. The `is_impossible` value shows whether the
existing annotation found relevant evidence for that category.

For example, `contract_0001__anti_assignment` has `is_impossible = False`,
meaning there is an annotation for Anti-Assignment. The related spans then
show the actual text that was identified as evidence. If `is_impossible =
True`, there is no corresponding answer span for that category.

For our review, we'll look at the Contract text in the spans and make our own
decision before comparing it with the existing annotation.

## 5. Blinded Review Set

Next, we have to create a review table containing the category and Contract evidence
needed to make an independent decision.

The existing `is_impossible` annotation will be retained separately and will
not be included in the initial review table. This prevents the existing
annotation from influencing the independent decision.

Each review item will have a unique review ID so that the independent
decision and subsequent comparison can be traced back to the original
Contract and annotation evidence.

In [1043]:
# Keep only annotation sets belonging to the 10 selected categories
review_annotation_sets = train_annotation_sets[
    train_annotation_sets["category_id"].isin(
        final_categories["category_id"]
    )
].copy()

# Add category names
review_annotation_sets = review_annotation_sets.merge(
    final_categories[["category_id", "category_name"]],
    on="category_id",
    how="left"
)

# Attach span text
review_evidence = train_spans.merge(
    review_annotation_sets[
        [
            "annotation_set_id",
            "contract_id",
            "category_id",
            "category_name",
            "is_impossible"
        ]
    ],
    on="annotation_set_id",
    how="inner"
)

# Keep the existing annotation separate
existing_annotations = review_evidence[
    [
        "annotation_set_id",
        "contract_id",
        "category_id",
        "category_name",
        "is_impossible"
    ]
].drop_duplicates()

# Create the blinded review table
blinded_review = review_evidence[
    [
        "span_id",
        "annotation_set_id",
        "contract_id",
        "category_id",
        "category_name",
        "answer_text",
        "answer_start",
        "answer_end"
    ]
].copy()

# Give each item a review ID
blinded_review.insert(
    0,
    "review_id",
    range(1, len(blinded_review) + 1)
)

print(f"Review evidence items: {len(blinded_review)}")
display(blinded_review.head())

Review evidence items: 3078


,review_id,span_id,annotation_set_id,contract_id,category_id,category_name,answer_text,answer_start,answer_end
0,1,contract_0001__renewal_term__span_000,contract_0001__renewal_term,contract_0001,renewal_term,Renewal Term,If Distributor comp...,5543,5881
1,2,contract_0001__governing_law__span_000,contract_0001__governing_law,contract_0001,governing_law,Governing Law,This Agreement is to be construed according to...,52061,52151
2,3,contract_0001__exclusivity__span_000,contract_0001__exclusivity,contract_0001,exclusivity,Exclusivity,The Company appoints the Distributor as an ex...,1854,2004
3,4,contract_0001__exclusivity__span_001,contract_0001__exclusivity,contract_0001,exclusivity,Exclusivity,Company hereby appoints ...,2112,2611
4,5,contract_0001__exclusivity__span_002,contract_0001__exclusivity,contract_0001,exclusivity,Exclusivity,The Distributor shall not order or ...,12390,12503


We now have 3,078 review evidence items from the 10 selected categories.
Each item connects a Contract to a category and includes the specific text
span that was identified as evidence. One thing to note: 3,078 is the number of evidence spans, not 3,078 unique Contracts.

Some Contracts have multiple spans for the same category, as we can see with
`contract_0001` and `exclusivity`. The `answer_start` and `answer_end` values
show where each span appears in the original Contract.

This gives us the text we can independently review without showing the
existing `is_impossible` decision.

### Review Evidence by Category

Before selecting examples for the independent review, I want to see how the
3,078 evidence items are distributed across the 10 categories. This will show
whether some categories have substantially more available evidence than others.

In [1044]:
review_evidence["category_name"].value_counts()

category_name
License Grant                  530
Cap On Liability               458
Audit Rights                   417
Anti-Assignment                408
Governing Law                  290
Exclusivity                    274
Revenue/Profit Sharing         269
Termination For Convenience    165
Renewal Term                   144
Uncapped Liability             123
Name: count, dtype: int64

The number of evidence spans varies across the 10 categories. License Grant
has the most evidence with 530 spans, while Uncapped Liability has the fewest
with 123.

This means the categories are not represented equally in the review set. For
the independent review, we'll sample examples from each category rather than
reviewing all 3,078 spans.

## 6. Selecting Evidence for Independent Review

I'm going to review 5 evidence spans from each of the 10 categories, for a total of
50 examples. A fixed random seed is used so that the same examples can be
reproduced later.

The existing annotation decision will remain hidden.

In [1045]:
review_sample = (
    blinded_review
    .groupby("category_name", group_keys=False)
    .sample(n=5, random_state=42)
    .sort_values(["category_name", "contract_id"])
    .reset_index(drop=True)
)

print(f"Total review examples: {len(review_sample)}")
print("\nExamples per category:")
print(review_sample["category_name"].value_counts())

display(
    review_sample[
        [
            "review_id",
            "category_name",
            "contract_id",
            "answer_text"
        ]
    ]
)

Total review examples: 50

Examples per category:
category_name
Anti-Assignment                5
Audit Rights                   5
Cap On Liability               5
Exclusivity                    5
Governing Law                  5
License Grant                  5
Renewal Term                   5
Revenue/Profit Sharing         5
Termination For Convenience    5
Uncapped Liability             5
Name: count, dtype: int64


,review_id,category_name,contract_id,answer_text
0,270,Anti-Assignment,contract_0042,This Agreement may not be assigned by Broker D...
1,337,Anti-Assignment,contract_0056,"The Distributor shall not sub-license, transfe..."
2,507,Anti-Assignment,contract_0073,Consequently either this Agreement or any of t...
3,1676,Anti-Assignment,contract_0214,"Either Party may, without consent of the other..."
4,1948,Anti-Assignment,contract_0249,Neither this Agreement nor any rights or oblig...
5,941,Audit Rights,contract_0137,Franchisee shall fully cooperate with Pretzel ...
6,1122,Audit Rights,contract_0160,In addition to any inspection rights granted u...
7,1296,Audit Rights,contract_0177,The Auditing Party may cause the Auditor to pe...
8,2441,Audit Rights,contract_0318,"SHPS shall have the right, upon reasonable pri..."
9,2725,Audit Rights,contract_0356,"Premier, shall have the right, directly or thr..."


## 7. Independent Review

For each example, I'm going to decide whether the Contract language
should count for the listed category.

I will record:
- **Decision:** Yes, No, or Unclear
- **Reasoning:** why the language does or does not fit the category

The existing annotation decision will be kept separate until after these
independent decisions are recorded.

In [1046]:
review_sample["my_decision"] = ""
review_sample["reasoning"] = ""

display(
    review_sample[
        [
            "review_id",
            "category_name",
            "contract_id",
            "answer_text",
            "my_decision",
            "reasoning"
        ]
    ]
)

,review_id,category_name,contract_id,answer_text,my_decision,reasoning
0,270,Anti-Assignment,contract_0042,This Agreement may not be assigned by Broker D...,,
1,337,Anti-Assignment,contract_0056,"The Distributor shall not sub-license, transfe...",,
2,507,Anti-Assignment,contract_0073,Consequently either this Agreement or any of t...,,
3,1676,Anti-Assignment,contract_0214,"Either Party may, without consent of the other...",,
4,1948,Anti-Assignment,contract_0249,Neither this Agreement nor any rights or oblig...,,
5,941,Audit Rights,contract_0137,Franchisee shall fully cooperate with Pretzel ...,,
6,1122,Audit Rights,contract_0160,In addition to any inspection rights granted u...,,
7,1296,Audit Rights,contract_0177,The Auditing Party may cause the Auditor to pe...,,
8,2441,Audit Rights,contract_0318,"SHPS shall have the right, upon reasonable pri...",,
9,2725,Audit Rights,contract_0356,"Premier, shall have the right, directly or thr...",,


# The Review
bear with me

## Category 1
Anti-Assignment

In [1047]:
display(review_sample['category_name'][0])
display(review_sample['answer_text'][0])

'Anti-Assignment'

'This Agreement may not be assigned by Broker Dealer without the prior written consent of Issuer and   Distributor, which shall not be unreasonably withheld.'

For the first sample, the category is Anti-Assignment which is a contract provision that restricts a party from transferring the contract, its rights, or its obligations to someone else without meeting certain conditions.

Since the answer_text talks about restricting the Broker Dealer from assigning
the Agreement without prior written consent from the Issuer and Distributor, I am classifying this sample's categorization as correct, Yes.

Next I input this into the table

In [1048]:
review_sample.loc[0, "my_decision"] = "Yes"

review_sample.loc[0, "reasoning"] = (
    "The provision explicitly restricts assignment of the Agreement without prior written consent."
)

In [1049]:
display(review_sample['category_name'][1])
display(review_sample['answer_text'][1])

'Anti-Assignment'

'The Distributor shall not sub-license, transfer or otherwise deal with the rights of use of the Trade Marks granted under this agreement.'

From here on out, a markdown box will only be created if needed

In [1050]:
review_sample.loc[1, "my_decision"] = "Yes"

review_sample.loc[1, "reasoning"] = (
    "The provision explicitly prohibits transferring or otherwise dealing with the rights granted under the agreement."
)

In [1051]:
display(review_sample['category_name'][2])
display(review_sample['answer_text'][2])

'Anti-Assignment'

'Consequently either this Agreement or any of the respective rights or obligations of the Parties hereunder may be assigned or otherwise transferred, in whole or in part, in any form whatsoever (including by way of change of Control), by either Party subject to the prior written consent of the other Party, which consent shall not be unreasonably withheld or delayed, and any attempt to do so without such consent shall be null and void.'

In [1052]:
review_sample.loc[2, "my_decision"] = "Yes"

review_sample.loc[2, "reasoning"] = (
    "The provision requires prior written consent before the Agreement or the Parties' rights or obligations can be assigned or transferred."
)

In [1053]:
display(review_sample['category_name'][3])
display(review_sample['answer_text'][3])

'Anti-Assignment'

'Either Party may, without consent of the other Party, assign this Agreement in whole to (i) in the case of RevMed, its successor in interest or assignee or purchaser, as applicable, in the case of a Change of Control or (ii) in the case of Sanofi, its successor in interest or assignee or purchaser, as applicable, in connection with the sale of all or substantially all of its assets to which this Agreement relates, or in connection with a merger, acquisition or similar transaction.'

In [1054]:
review_sample.loc[3, "my_decision"] = "Yes"

review_sample.loc[3, "reasoning"] = (
    "The provision directly addresses assignment of the Agreement and specifies circumstances where assignment may occur without consent."
)

In [1055]:
display(review_sample['category_name'][4])
display(review_sample['answer_text'][4])

'Anti-Assignment'

"Neither this Agreement nor any rights or obligations hereunder shall be assignable by a Party without the prior written consent of the other Party, provided that either Party shall have the right, on notice to but without the other Party's consent, to assign this Agreement and its rights and obligations contained herein, to an affiliate or to a third party who is not a competitor of the other Party in connection with a sale of all or substantially all of the assigning Party's business or assets relating to this Agreement."

In [1056]:
review_sample.loc[4, "my_decision"] = "Yes"

review_sample.loc[4, "reasoning"] = (
    "The provision explicitly restricts assignment of the Agreement and "
    "related rights or obligations without prior written consent, while "
    "providing specific exceptions."
)

## Category 2
Audit Rights


We're now onto the second category, Audit Rights. Audit Rights are contract provisions that give one party the right to inspect or review the other party’s records, books, accounts, or documents to verify compliance with the agreement.

For example, this sample talks about audits and how franchises who work with Pretzel Time must cooperate with and audits done by Pretzel Time.

In [1057]:
display(review_sample['category_name'][5])
display(review_sample['answer_text'][5])

'Audit Rights'

"Franchisee shall fully cooperate with Pretzel Time's representatives and independent accountants hired by Pretzel Time to conduct any such inspection or audit."

In [1058]:
review_sample.loc[5, "my_decision"] = "Yes"

review_sample.loc[5, "reasoning"] = (
    "The provision explicitly refers to an inspection or audit conducted by the other party's representatives and independent accountants."
)

In [1059]:
display(review_sample['category_name'][6])
display(review_sample['answer_text'][6])

'Audit Rights'

'In addition to any inspection rights granted under Law, upon notice to the Company of at least twenty-four (24) hours, each Party shall have full access to all properties, books of account, and records of the Company.'

In [1060]:
review_sample.loc[6, "my_decision"] = "Yes"

review_sample.loc[6, "reasoning"] = (
    "The provision gives each Party access to the Company's properties, books, and records, which establishes an inspection or audit right."
)

In [1061]:
display(review_sample['category_name'][7])
display(review_sample['answer_text'][7])

'Audit Rights'

'The Auditing Party may cause the Auditor to perform such an audit not more than once in any 12-month period, unless a prior audit within the past two years revealed that the amount owed by the Audited Party to the Auditing Party was underpaid in excess of 8% of the amount owed, in which case an audit may be performed no more frequently than twice in any 12-month period.'

In [1062]:
review_sample.loc[7, "my_decision"] = "Yes"

review_sample.loc[7, "reasoning"] = (
    "The provision explicitly grants and regulates the right to perform audits, including limits on how frequently audits may occur."
)

In [1063]:
display(review_sample['category_name'][8])
display(review_sample['answer_text'][8])

'Audit Rights'

'SHPS shall have the right, upon reasonable prior written notice, to examine, copy and audit such records. Such audit shall be conducted at the location where such records are maintained and shall be at the expense of SHPS.'

In [1064]:
review_sample.loc[8, "my_decision"] = "Yes"

review_sample.loc[8, "reasoning"] = (
    "The provision explicitly gives SHPS the right to examine, copy, and audit records upon reasonable prior written notice."
)

In [1065]:
display(review_sample['category_name'][9])
display(review_sample['answer_text'][9])

'Audit Rights'

'Premier, shall have the right, directly or through its representative, to inspect, copy, and audit all such records upon reasonable request and during normal business hours, acknowledging that access to accounting and purchasing records will be limited to those supporting pass-through materials costs and purchases of Premier specified equipment if any.'

In [1066]:
review_sample.loc[9, "my_decision"] = "Yes"

review_sample.loc[9, "reasoning"] = (
    "The provision explicitly gives Premier the right to inspect, copy, and audit records, with specified limits on the records that can be accessed."
)

## Category 3
Cap On Liability

In [1067]:
display(review_sample['category_name'][10])
display(review_sample['answer_text'][10])

'Cap On Liability'

'If a Buyer receives a product that fails to conform to these representations and warranties, the sole remedies of Buyer for the breach of warranty will be to: (1) reject and return the non-conforming product to Seller for a refund or credit, or a replacement conforming product, in the manner and time period provided in the SOP; (2) obtain reimbursement from Seller for actual, reasonable, substantiated out-of-pocket expenses incurred by Buyer in the recovery, return or disposal of a non-conforming product that is the subject of a mandatory product recall required under Applicable Laws or a voluntary withdrawal declared by Seller or approved by Seller (such approval not to be unreasonably withheld, conditioned or delayed); and (3) obtain indemnification from Seller for any Indemnified Claim arising from or related to the non-conforming product as provided in Section 7.'

Now onto the third category, Cap on Liability. Cap on Liability is when a contract limits the maximum amount of money a party can be liable for if something goes wrong.

So for this example it doesn't talk about a set limit of money that may be refunded or credited, so shoudl be classified as No.

In [1068]:
review_sample.loc[10, "my_decision"] = "No"

review_sample.loc[10, "reasoning"] = (
    "The provision describes the Buyer's remedies for a non-conforming product but does not establish a maximum limit or cap on liability."
)

In [1069]:
display(review_sample['category_name'][11])
display(review_sample['answer_text'][11])

'Cap On Liability'

'IBM will give Customer a credit equal to the amount Customer paid IBM for the applicable Materials or for use of the applicable Base Components up to a maximum of twelve (12) months of applicable charges.'

In [1070]:
review_sample.loc[11, "my_decision"] = "Yes"

review_sample.loc[11, "reasoning"] = (
    "The provision sets a maximum amount for the credit that IBM will provide, limiting it to twelve months of applicable charges."
)

In [1071]:
display(review_sample['category_name'][12])
display(review_sample['answer_text'][12])

'Cap On Liability'

'Except for claims arising under section 6, in no event will either party be liable for any special, indirect, incidental or consequential damages.'

In [1072]:
review_sample.loc[12, "my_decision"] = "Yes"

review_sample.loc[12, "reasoning"] = (
    "The provision limits liability by excluding special, indirect, incidental, and consequential damages, although it does not specify a monetary cap."
)

In [1073]:
display(review_sample['category_name'][13])
display(review_sample['answer_text'][13])

'Cap On Liability'

'The warranty and remedies set forth in Exhibit B are exclusive and in lieu of any other warranties or remedies, express or implied, including the implied warranties of merchantability and fitness for intended or particular purpose.'

In [1074]:
review_sample.loc[13, "my_decision"] = "No"

review_sample.loc[13, "reasoning"] = (
    "The provision makes certain warranties and remedies exclusive, but it does not establish a limit or cap on liability."
)

In [1075]:
display(review_sample['category_name'][14])
display(review_sample['answer_text'][14])

'Cap On Liability'

'In addition, in the event of a material breach by Nexstar of its obligations hereunder, WYZZ shall be entitled to terminate this Agreement and exercise its rights pursuant to Section 25(a) hereof (except that WYZZ may not assert consequential, special or punitive damages or any claim for lost profits).'

In [1076]:
review_sample.loc[14, "my_decision"] = "Yes"

review_sample.loc[14, "reasoning"] = (
    "The provision limits available damages by excluding consequential, special, and punitive damages and claims for lost profits."
)

## Category 4
Exclusivity

In [1077]:
display(review_sample['category_name'][15])
display(review_sample['answer_text'][15])

'Exclusivity'

'In consideration of the exclusivity rights granted to LEA, commencing with the seventh (7t h) month of the Term and continuing each year of the Term thereafter, the minimum Royalties payable to T&B each month shall be the greater of the (i) applicable monthly Base Royalty and Marketing Royalty or (ii) $200,000.'

The fourth category is exclusivity which means a contract gives one party exclusive rights to do something, or restricts a party from working with or doing business with others.

For this example this provision explicitly talks about exclusivity rights and so can be classified yes.

In [1078]:
review_sample.loc[15, "my_decision"] = "Yes"

review_sample.loc[15, "reasoning"] = (
    "The provision explicitly refers to exclusivity rights granted to LEA, indicating that LEA receives exclusive rights under the agreement."
)

In [1079]:
display(review_sample['category_name'][16])
display(review_sample['answer_text'][16])

'Exclusivity'

'Reseller shall not obtain the           TouchStar Software or Support Services (or any software or services           which compete with the TouchStar Software) for sale from any Entity           other than TouchStar or its authorized agents.'

In [1080]:
review_sample.loc[16, "my_decision"] = "Yes"

review_sample.loc[16, "reasoning"] = (
    "The provision restricts the Reseller from obtaining the software or competing services from any entity other than TouchStar or its authorized agents."
)

In [1081]:
display(review_sample['category_name'][17])
display(review_sample['answer_text'][17])

'Exclusivity'

'Nantz Communications and Nantz expressly agree that the Endorsement will not be granted to anyone other than the Company for use during the Term in connection with the advertisement and promotion of sportswear apparel, hats and shoes.'

In [1082]:
review_sample.loc[17, "my_decision"] = "Yes"

review_sample.loc[17, "reasoning"] = (
    "The provision explicitly prevents the endorsement from being granted to anyone other than the Company during the Term."
)

In [1083]:
display(review_sample['category_name'][18])
display(review_sample['answer_text'][18])

'Exclusivity'

'During the Term of this Agreement, except as otherwise permitted by this Section 3(a)(v), VS agrees that it shall not enter into the same or substantially similar Commitments with any other company or entity which performs clinical research services the same or similar to those provided by PPD or any PPD affiliate (collectively, "PPD Competitor"), nor shall VS provide preferred pricing to a PPD Competitor which is better than that provided by VS hereunder to PPD.'

In [1084]:
review_sample.loc[18, "my_decision"] = "Yes"

review_sample.loc[18, "reasoning"] = (
    "The provision restricts VS from entering into similar commitments with PPD competitors and from providing them with better pricing."
)

In [1085]:
display(review_sample['category_name'][19])
display(review_sample['answer_text'][19])

'Exclusivity'

'The Company hereby appoints and grants Distributor the exclusive right to sell the products of the Company, including the Snotarator™ Nasal Aspirator,  ("Products") listed in the current "Price List" (Exhibit "A" attached hereto).'

In [1086]:
review_sample.loc[19, "my_decision"] = "Yes"

review_sample.loc[19, "reasoning"] = (
    "The provision explicitly grants the Distributor the exclusive right to sell the Company's products."
)

## Category 5
Governing Law

In [1087]:
display(review_sample['category_name'][20])
display(review_sample['answer_text'][20])

'Governing Law'

'This Amendment shall be governed by and construed in accordance with the laws of Japan.'

Onto the next category, Governing Law. Governing Law is Governing Law is the contract provision that specifies which state, country, or jurisdiction's laws will be used to interpret and enforce the agreement.

For this example, the provision speaks about being in accordance with the laws of Japan, so can be classified Yes.

In [1088]:
review_sample.loc[20, "my_decision"] = "Yes"

review_sample.loc[20, "reasoning"] = (
    "The provision explicitly states that the Amendment is governed by and construed in accordance with the laws of Japan."
)

In [1089]:
display(review_sample['category_name'][21])
display(review_sample['answer_text'][21])

'Governing Law'

'This Agreement shall be governed by and construed in accordance with the laws of the State of California without regard to its conflict of laws provisions.'

In [1090]:
review_sample.loc[21, "my_decision"] = "Yes"

review_sample.loc[21, "reasoning"] = (
    "The provision explicitly states that the Agreement is governed by and construed in accordance with California law."
)

In [1091]:
display(review_sample['category_name'][22])
display(review_sample['answer_text'][22])

'Governing Law'

'This Agreement shall be governed by and construed in accordance with the laws of New York, US, without reference to its conflict of laws principles, and shall not be governed by the United Nations Convention of International Contracts on the Sale of Goods (the Vienna Convention).'

In [1092]:
review_sample.loc[22, "my_decision"] = "Yes"

review_sample.loc[22, "reasoning"] = (
    "The provision explicitly states that the Agreement is governed by and construed in accordance with the laws of New York."
)

In [1093]:
display(review_sample['category_name'][23])
display(review_sample['answer_text'][23])

'Governing Law'

'This Agreement shall be governed by and construed in accordance with the laws of the Province of Ontario and the federal laws of Canada applicable in the Province of Ontario.'

In [1094]:
review_sample.loc[23, "my_decision"] = "Yes"

review_sample.loc[23, "reasoning"] = (
    "The provision explicitly states that the Agreement is governed by Ontario law and applicable federal laws of Canada."
)

In [1095]:
display(review_sample['category_name'][24])
display(review_sample['answer_text'][24])

'Governing Law'

'This Agreement will be governed by, construed and enforced in accordance with the laws of the State of Texas.'

In [1096]:
review_sample.loc[24, "my_decision"] = "Yes"

review_sample.loc[24, "reasoning"] = (
    "The provision explicitly states that the Agreement is governed by, construed, and enforced under Texas law."
)

## Category 6
License Grant

In [1097]:
display(review_sample['category_name'][25])
display(review_sample['answer_text'][25])

'License Grant'

'Develop, Manufacture, or Commercialize the Product for use outside the Licensed Field.'

Category six is License Grant which means one party gives another party permission to use certain intellectual property or other protected material under specified conditions.

For this example, while License is discussed, nothing is granted, so should be classified No.

In [1098]:
review_sample.loc[25, "my_decision"] = "No"

review_sample.loc[25, "reasoning"] = (
    "The provision describes activities involving the Product outside the "
    "Licensed Field, but it does not itself grant a license or permission "
    "to use intellectual property."
)

In [1099]:
display(review_sample['category_name'][26])
display(review_sample['answer_text'][26])

'License Grant'

'Franchisee  agrees and grants to Pretzel Time and its Affiliates a perpetual and worldwide  right to use and  authorize  other  Pretzel  Time Units or other food service businesses  operated by Pretzel Time or its Affiliates,  franchisees and designees  to  use  such  ideas,  recipes,  formulas,   concepts,  methods,  and techniques  relating to the development  and/or  operation of a dessert or snack food business.'

In [1100]:
review_sample.loc[26, "my_decision"] = "Yes"

review_sample.loc[26, "reasoning"] = (
    "The provision explicitly grants Pretzel Time and its Affiliates a perpetual and worldwide right to use the specified ideas, recipes, formulas, concepts, methods, and techniques."
)

In [1101]:
display(review_sample['category_name'][27])
display(review_sample['answer_text'][27])

'License Grant'

"As part of the exclusive distribution right granted in this Section 2, Vendor hereby grants Distributor the non- exclusive, non-transferable right to use and display Vendor's trademarks, logos, Product photographs and images, Product advertising and promotional copy, including but not limited to the materials contained in Vendor's website, in connection with the promotion, advertising and distribution of the Products."

In [1102]:
review_sample.loc[27, "my_decision"] = "Yes"

review_sample.loc[27, "reasoning"] = (
    "The provision explicitly grants Distributor the right to use and display the Vendor's trademarks, logos, images, and promotional materials."
)

In [1103]:
display(review_sample['category_name'][28])
display(review_sample['answer_text'][28])

'License Grant'

'During the term of this Agreement, and subject to the terms and conditions hereof, STAAR hereby grants to Distributor, and Distributor hereby accepts, the limited, nontransferable, nonexclusive right and license to use the trade name, trademarks, and logos of STAAR (collectively, "Trademarks"), without the right to grant sublicenses, solely in connection with the marketing, distribution and sale of the Products in the Territory pursuant to this Agreement.'

In [1104]:
review_sample.loc[28, "my_decision"] = "Yes"

review_sample.loc[28, "reasoning"] = (
    "The provision explicitly grants Distributor a limited, nontransferable, nonexclusive right and license to use STAAR's trade name, trademarks, and logos."
)

In [1105]:
display(review_sample['category_name'][29])
display(review_sample['answer_text'][29])

'License Grant'

'Subject to the terms and conditions hereof, drkoop.com hereby represents that it has the power and authority to grant, and does hereby grant to Sponsor a non-exclusive, non-transferable, royalty-free, worldwide license to reproduce and display all logos, trademarks, trade names and similar identifying material relating to drkoop.com and, solely as allowed pursuant to this Agreement, to the Dr. C. Everett Koop name (collectively, the "drkoop.com Marks") solely in connection with the promotion, marketing and distribution of the parties and the Sites in accordance with the terms hereof, provided, however, that Sponsor shall, other than as specifically provided for in Section 4.4 of this Agreement, not make any specific use of any drkoop.com Marks without first submitting a sample of such use to drkoop.com and obtaining its prior consent, which consent shall not be unreasonably withheld.'

In [1106]:
review_sample.loc[29, "my_decision"] = "Yes"

review_sample.loc[29, "reasoning"] = (
    "The provision explicitly grants Sponsor a non-exclusive, non-transferable, royalty-free, worldwide license to reproduce and display the specified trademarks, trade names, and other identifying materials."
)

## Category 7
Renewal Term

In [1107]:
display(review_sample['category_name'][30])
display(review_sample['answer_text'][30])

'Renewal Term'

'As provided for in this Section 1, the term of this Agreement shall be for a period of five (5) years, beginning on the Effective Date (the "Initial Term"); provided, however, the Initial Term shall be subject to automatic successive renewal terms of three (3) years each (the "Renewal Terms" and together with the Initial Term, the "Term").'

The next category is Renewal Term. Renewal Term is language about whether and how the contract continues after its initial term ends.

For this example, the provision explicitly talks about renewal term, classified as Yes.

In [1108]:
review_sample.loc[30, "my_decision"] = "Yes"
review_sample.loc[30, "reasoning"] = (
    "The provision explicitly states that the Initial Term is subject to automatic successive renewal terms of three years each, directly describing how the Agreement continues after the initial term."
)

In [1109]:
display(review_sample['category_name'][31])
display(review_sample['answer_text'][31])

'Renewal Term'

'This Agreement may be renewed for additional periods of one (1) year (each such additional period, a "Renewal Term") unless either Party provides notice of nonrenewal upon not less than [***] prior written notice to the other Party.'

In [1110]:
review_sample.loc[31, "my_decision"] = "Yes"
review_sample.loc[31, "reasoning"] = (
    "The provision explicitly states that the Agreement may be renewed for additional one-year periods and defines each additional period as a 'Renewal Term.' It also describes the notice required to prevent renewal."
)

In [1111]:
display(review_sample['category_name'][32])
display(review_sample['answer_text'][32])

'Renewal Term'

'Within ninety (90) days of our receipt of your notice to renew, we will furnish you with written notice of:  (i) reasons which could cause us not to grant a renewal to you including but not limited to any deficiencies which require correction and a schedule for correction by you; and (ii) our then-current requirements relating to the image, appearance, decoration, furnishing, equipping and stocking of Buffalo Wild Wings businesses, and a schedule for effecting upgrading or modifications in order to bring the Franchised Restaurant in compliance, as a condition of renewal.  Renewal of the franchise shall be conditioned upon your compliance with such requirements and continued compliance with all the terms and conditions of this Agreement up to the date of termination of the initial term.'

In [1112]:
review_sample.loc[32, "my_decision"] = "Yes"
review_sample.loc[32, "reasoning"] = (
    "The provision directly addresses renewal of the franchise and describes conditions that must be satisfied for renewal, including compliance with specified requirements and the terms of the Agreement through the end of the initial term."
)

In [1113]:
display(review_sample['category_name'][33])
display(review_sample['answer_text'][33])

'Renewal Term'

'This Agreement will automatically be renewed for periods of       twelve (12) months unless either Party gives six (6) months written       notice of its intent to terminate this Agreement.'

In [1114]:
review_sample.loc[33, "my_decision"] = "Yes"
review_sample.loc[33, "reasoning"] = (
    "The provision explicitly states that the Agreement automatically renews for successive twelve-month periods unless either Party gives the required notice to terminate."
)

In [1115]:
display(review_sample['category_name'][34])
display(review_sample['answer_text'][34])

'Renewal Term'

'This Agreement shall be subject to an automatic extension for consecutive one (1) year periods  thereafter (each, an "Extension Term" and together with the Initial Term, the "Term"), unless terminated (i) in accordance with its terms or (ii)  upon thirty (30) days\' written notice by either Party to the other Party.'

In [1116]:
review_sample.loc[34, "my_decision"] = "Yes"
review_sample.loc[34, "reasoning"] = (
    "The provision states that the Agreement automatically extends for consecutive one-year periods after the Initial Term. These extensions describe the continuation of the Agreement and function as renewal terms."
)

## Category 8
Revenue/Profit Sharing

In [1117]:
display(review_sample['category_name'][35])
display(review_sample['answer_text'][35])

'Revenue/Profit Sharing'

'HSNS agrees to pay E.piphany an additional                   $0.005 per email for any email distributed by HSNS as a result                   of any deal it closes that either results from a lead                   generated by E.piphany or in which E.piphany assisted prior to                   closing for the first year after the deal closes.'

Category 8 is Revenue/Profit Sharing which means contract language that
requires or describes one party sharing revenue, profits, sales, or other
financial proceeds with another party.

In this example we don't see any discussion of revenue or profit sharing, just a fee between companies which is not the same, so can be classified No.

In [1118]:
review_sample.loc[35, "my_decision"] = "No"
review_sample.loc[35, "reasoning"] = (
    "The provision requires HSNS to pay E.piphany an additional fixed amount per email for certain deals. It describes a fee or payment obligation, but does not establish sharing of revenue or profit between the parties."
)

In [1119]:
display(review_sample['category_name'][36])
display(review_sample['answer_text'][36])

'Revenue/Profit Sharing'

'During the Distribution Term, and in addition to the consideration provided pursuant to Sections 6.1, 6.2, 6.3, and 6.4, for all Product supplied by Zogenix to Distributor under purchase orders submitted pursuant to the Supply Agreement in a particular Fiscal Year, Distributor shall pay to Zogenix a transfer price per unit of Product supplied (the "Transfer Price") equal to the sum of (i) [***] of the Fully-Burdened Manufacturing Cost per unit of Product for such Fiscal Year, (ii) [***] of aggregate annual Net Sales for such Fiscal Year, and (iii) the applicable markup percent of the applicable aggregate Net Price for such Fiscal Year, which markup percent is determined based on the incremental amount of Product ordered in such Fiscal Year as set forth below, as may be adjusted pursuant to Section 6.5(b):\n\nAmount of Product Supplied per Fiscal Year Net Price Markup\n\nFor the portion of Product supplied less than or equal to the equivalent of [***] in Net Sales in such Fiscal Year [

In [1120]:
review_sample.loc[36, "my_decision"] = "Yes"
review_sample.loc[36, "reasoning"] = (
    "The Transfer Price is calculated partly using aggregate annual Net Sales, so the payment to Zogenix is tied to revenue generated from the Product. Although the provision is framed as a transfer-price formula rather than explicitly calling it revenue sharing, its calculation based on Net Sales makes it relevant to Revenue/Profit Sharing."
)

In [1121]:
display(review_sample['category_name'][37])
display(review_sample['answer_text'][37])

'Revenue/Profit Sharing'

'In consideration for the intangible rights granted hereunder, for each Year in which the Spoken-Word Audio Sub-Section (including the Mirror Company Site) generates revenue of at [***] (the "Revenue Threshold"), Company will pay ACSI a royalty equal to [***] of all revenues generated from the Spoken-Word Audio Sub-Section (including, for the avoidance of doubt, any revenue received by Company from any Company customer who first links to the Mirror Company Site from the Spoken-Word Audio Sub-Section and who later accesses the Company Site directly) in excess of Revenue Threshold (the "Royalties") for each Year of the Term.'

In [1122]:
review_sample.loc[37, "my_decision"] = "Yes"
review_sample.loc[37, "reasoning"] = (
    "The provision explicitly requires the Company to pay ACSI a royalty calculated as a percentage of revenues generated above a specified Revenue Threshold. This directly describes revenue sharing between the parties."
)

In [1123]:
display(review_sample['category_name'][38])
display(review_sample['answer_text'][38])

'Revenue/Profit Sharing'

'In consideration of the license of Liquidmetal Technical Information and the Licensed Equipment granted by Liquidmetal, Eutectix agrees to pay Liquidmetal a cash royalty based on a percentage of the invoice price of any Licensed Products (but not including Liquidmetal Products) sold by Eutectix or its permitted sublicensees and for which payment was actually received by Eutectix.'

In [1124]:
review_sample.loc[38, "my_decision"] = "Yes"
review_sample.loc[38, "reasoning"] = (
    "The provision requires Eutectix to pay Liquidmetal a royalty based on a percentage of the invoice price of Licensed Products sold. Because the payment is calculated as a percentage of sales, it directly involves sharing revenue generated from the Licensed Products."
)

In [1125]:
display(review_sample['category_name'][39])
display(review_sample['answer_text'][39])

'Revenue/Profit Sharing'

'In addition, Client shall be entitled to receive a royalty payment on the shipping and  handling charges paid by customers during the applicable Calendar Quarter ("Shipping Royalty") equal to the Royalty percentage  multiplied by the shipping profit.'

In [1126]:
review_sample.loc[39, "my_decision"] = "Yes"
review_sample.loc[39, "reasoning"] = (
    "The provision explicitly establishes a royalty payment calculated using a royalty percentage multiplied by shipping profit. Because the payment is directly based on profit, it clearly falls under Revenue/Profit Sharing."
)

## Category 9
Termination For Convenience

In [1127]:
display(review_sample['category_name'][40])
display(review_sample['answer_text'][40])

'Termination For Convenience'

'Party A is entitled to unilaterally terminate this Agreement within three natural months from the signing date of this Agreement.'

For Category 9, we are focusing on Termination For Convenience which is when a contract can be ended without requiring the other party to have breached the agreement or some other specific reason.

For this example, notice that party A is entitled to unilaterally terminate and agreement within 3 months, meaning that as long as it is within 3 months of signing, they can terminate with convenience.

In [1128]:
review_sample.loc[40, "my_decision"] = "Yes"
review_sample.loc[40, "reasoning"] = (
    "The provision gives Party A the unilateral right to terminate the Agreement within three months of signing without stating that a breach or other specific cause is required. This indicates a termination right that can be exercised for convenience."
)

In [1129]:
display(review_sample['category_name'][41])
display(review_sample['answer_text'][41])

'Termination For Convenience'

'The Agreement rests, for all that, cancellable at any time by any of the parties before the expiry date of the Agreement or any of itsrenewals, upon three months prior written notice.'

In [1130]:
review_sample.loc[41, "my_decision"] = "Yes"
review_sample.loc[41, "reasoning"] = (
    "The provision allows any party to cancel the Agreement at any time before its expiration or renewal, subject only to three months' prior written notice. It does not require a breach or other specific cause, so it represents termination for convenience."
)

In [1131]:
display(review_sample['category_name'][42])
display(review_sample['answer_text'][42])

'Termination For Convenience'

'Written notice of intention to withdraw must be served in writing upon the remaining Participants at least Thirty (30) business days prior to the withdrawal date.'

In [1132]:
review_sample.loc[42, "my_decision"] = "Unclear"
review_sample.loc[42, "reasoning"] = (
    "The provision describes a right to withdraw with thirty days' written notice, but the excerpt does not explain whether the withdrawal can occur without cause or is subject to conditions elsewhere in the Agreement. Additional context would be needed to determine whether this is termination for convenience."
)

In [1133]:
display(review_sample['category_name'][43])
display(review_sample['answer_text'][43])

'Termination For Convenience'

"Licensee may terminate this Agreement for convenience upon eighteen (18) months' advance written notice to Bioeq; provided, however, that any such termination for convenience shall not become effective prior to twelve (12) months after the First Commercial Sale of the first Licensed Product."

In [1134]:
review_sample.loc[43, "my_decision"] = "Yes"
review_sample.loc[43, "reasoning"] = (
    "The provision explicitly states that Licensee may terminate the Agreement for convenience upon eighteen months' advance written notice. It also specifies a timing condition for when the termination can become effective."
)

In [1135]:
display(review_sample['category_name'][44])
display(review_sample['answer_text'][44])

'Termination For Convenience'

'Notwithstanding the foregoing, either party may terminate this Agreement at any time without liability by providing one hundred eighty (180) days written notice to the other party.'

In [1136]:
review_sample.loc[44, "my_decision"] = "Yes"
review_sample.loc[44, "reasoning"] = (
    "The provision allows either party to terminate the Agreement at any time without liability, subject only to 180 days' written notice. It does not require a breach or other specific cause, so it represents termination for convenience."
)

## Category 10
Uncapped Liability

In [1137]:
display(review_sample['category_name'][45])
display(review_sample['answer_text'][45])

'Uncapped Liability'

'EXCEPT IN CONNECTION WITH A BREACH BY EITHER PARTY OF ARTICLE 9 OR SECTION 10.1.4  [Representations and Warranties] (v) AND THE INDEMNIFICATION OBLIGATIONS OF LEADERSONLINE UNDER SECTION 11.4(i)(c)  [Indemnification by LeadersOnline] AND THE INDEMNIFICATION OBLIGATIONS OF VERTICALNET UNDER SECTION 11.5(i)(c)  [Indemnification by VerticalNet], NEITHER PARTY WILL BE LIABLE FOR ANY SPECIAL, INDIRECT, CONSEQUENTIAL, EXEMPLARY OR INCIDENTAL DAMAGES ARISING OUT OF OR RELATED TO THIS AGREEMENT, HOWEVER CAUSED AND UNDER ANY THEORY'

Our final category, Uncapped Liability, refers to when a party's liability is not subject to the usual monetary or other liability cap.

For this example, the provision creates an exception to a limitation on liability for certain breaches and indemnification obligations. These exceptions mean that the specified liabilities are not subject to the stated damages limitation. However, the provision does not explicitly state that these liabilities are unlimited, so this type of language can be difficult to distinguish from a general limitation-of-liability provision.

In [1138]:
review_sample.loc[45, "my_decision"] = "Yes"
review_sample.loc[45, "reasoning"] = (
    "The provision establishes a limitation on liability for special, indirect, consequential, exemplary, and incidental damages, but expressly excludes specified breaches and indemnification obligations from that limitation. These exceptions indicate that the listed liabilities are not subject to the stated liability restriction."
)

In [1139]:
display(review_sample['category_name'][46])
display(review_sample['answer_text'][46])

'Uncapped Liability'

"In no event will either party be liable to the other for special, incidental, or indirect damages or for any consequential damages (including lost profits or savings), even if they are informed of the possibility; provided that this Section 10.0 does not apply to Customer's failure to pay any amounts owing to IBM under this Agreement (including amounts owing for Services that would have been rendered but for Customer's breach of this Agreement)."

In [1140]:
review_sample.loc[46, "my_decision"] = "Yes"
review_sample.loc[46, "reasoning"] = (
    "The provision excludes certain types of damages from liability but explicitly states that the limitation does not apply to Customer's failure to pay amounts owed to IBM. This creates an exception to the liability limitation, although the provision does not explicitly state that the payment obligation is unlimited."
)

In [1141]:
display(review_sample['category_name'][47])
display(review_sample['answer_text'][47])

'Uncapped Liability'

"With the exception of wilful misconduct by a Party, and such cases where a limitation of liability and/or indemnification is not possible under applicable law, for which cases there shall be no limitation, any and all liability and/or indemnification obligations of each of BII and XENCOR under this Agreement shall be:   a. excluded for incidental, indirect, consequential, punitive or special damages (provided that the foregoing shall not exclude a Party's right to consequential or incidental"

In [1142]:
review_sample.loc[47, "my_decision"] = "Yes"
review_sample.loc[47, "reasoning"] = (
    "The provision explicitly states that there shall be no limitation of liability for wilful misconduct and for cases where a limitation of liability or indemnification is not possible under applicable law. These are explicit exceptions to the liability limitations."
)

In [1143]:
display(review_sample['category_name'][48])
display(review_sample['answer_text'][48])

'Uncapped Liability'

'EXCEPT FOR LIABILITY ARISING FROM SECTION 9.3  [Intellectual Property Infringement], IN NO EVENT SHALL EITHER PARTY BE LIABLE UNDER THIS AGREEMENT FOR AN AMOUNT GREATER THAN THE AMOUNT THAT SUCH PARTY HAS EARNED PURSUANT TO THE REVENUE SHARING PROVISIONS OF SECTION 5.4  [Share of Net Revenue] IN THE TWELVE MONTH PERIOD PRECEDING THE CLAIM.'

In [1144]:
review_sample.loc[48, "my_decision"] = "Yes"
review_sample.loc[48, "reasoning"] = (
    "The provision establishes a general liability cap based on the amount earned through revenue sharing, but explicitly excludes liability arising from Intellectual Property Infringement from that cap. This means the specified IP infringement liability is not subject to the stated limit."
)

In [1145]:
display(review_sample['category_name'][49])
display(review_sample['answer_text'][49])

'Uncapped Liability'

"Subject to Section 17(c), in no event shall either Party's liability under this Agreement exceed the aggregate of all amounts paid under this Agreement and amounts that have accrued but not yet been paid in the twelve (12) months preceding the event giving rise to the claim."

In [1146]:
review_sample.loc[49, "my_decision"] = "No"
review_sample.loc[49, "reasoning"] = (
    "The provision explicitly establishes a maximum liability amount based on payments made or accrued during the preceding twelve months. Although Section 17(c) may contain an exception, this excerpt itself describes a liability cap rather than uncapped liability."
)

## Full Table Review

In [1147]:
display(
    review_sample[
        [
            "review_id",
            "category_name",
            "contract_id",
            "answer_text",
            "my_decision",
            "reasoning"
        ]
    ]
)

,review_id,category_name,contract_id,answer_text,my_decision,reasoning
0,270,Anti-Assignment,contract_0042,This Agreement may not be assigned by Broker D...,Yes,The provision explicitly restricts assignment ...
1,337,Anti-Assignment,contract_0056,"The Distributor shall not sub-license, transfe...",Yes,The provision explicitly prohibits transferrin...
2,507,Anti-Assignment,contract_0073,Consequently either this Agreement or any of t...,Yes,The provision requires prior written consent b...
3,1676,Anti-Assignment,contract_0214,"Either Party may, without consent of the other...",Yes,The provision directly addresses assignment of...
4,1948,Anti-Assignment,contract_0249,Neither this Agreement nor any rights or oblig...,Yes,The provision explicitly restricts assignment ...
5,941,Audit Rights,contract_0137,Franchisee shall fully cooperate with Pretzel ...,Yes,The provision explicitly refers to an inspecti...
6,1122,Audit Rights,contract_0160,In addition to any inspection rights granted u...,Yes,The provision gives each Party access to the C...
7,1296,Audit Rights,contract_0177,The Auditing Party may cause the Auditor to pe...,Yes,The provision explicitly grants and regulates ...
8,2441,Audit Rights,contract_0318,"SHPS shall have the right, upon reasonable pri...",Yes,The provision explicitly gives SHPS the right ...
9,2725,Audit Rights,contract_0356,"Premier, shall have the right, directly or thr...",Yes,The provision explicitly gives Premier the rig...


In [1148]:
print(review_sample["my_decision"].value_counts())

my_decision
Yes        44
No          5
Unclear     1
Name: count, dtype: int64


# Compare to Existing Annotations
This is where I'm going to compare my decision of if these categories are correct to the preexisting decisions. This allows me to identify agreements, disagreements, and potentially unclear annotations.

In [1149]:
review_results = review_sample.merge(
    existing_annotations[
        [
            "contract_id",
            "category_id",
            "category_name",
            "is_impossible"
        ]
    ],
    on=["contract_id", "category_name"],
    how="left"
)
 #NOTE HERE IS WHERE THE PREVIOUS RESULTS COME FROM
review_results["existing_decision"] = review_results["is_impossible"].map({
    True: "No",
    False: "Yes"
})

display(
    review_results[
        [
            "review_id",
            "category_name",
            "contract_id",
            "answer_text",
            "my_decision",
            "existing_decision",
            "reasoning"
        ]
    ]
)

,review_id,category_name,contract_id,answer_text,my_decision,existing_decision,reasoning
0,270,Anti-Assignment,contract_0042,This Agreement may not be assigned by Broker D...,Yes,Yes,The provision explicitly restricts assignment ...
1,337,Anti-Assignment,contract_0056,"The Distributor shall not sub-license, transfe...",Yes,Yes,The provision explicitly prohibits transferrin...
2,507,Anti-Assignment,contract_0073,Consequently either this Agreement or any of t...,Yes,Yes,The provision requires prior written consent b...
3,1676,Anti-Assignment,contract_0214,"Either Party may, without consent of the other...",Yes,Yes,The provision directly addresses assignment of...
4,1948,Anti-Assignment,contract_0249,Neither this Agreement nor any rights or oblig...,Yes,Yes,The provision explicitly restricts assignment ...
5,941,Audit Rights,contract_0137,Franchisee shall fully cooperate with Pretzel ...,Yes,Yes,The provision explicitly refers to an inspecti...
6,1122,Audit Rights,contract_0160,In addition to any inspection rights granted u...,Yes,Yes,The provision gives each Party access to the C...
7,1296,Audit Rights,contract_0177,The Auditing Party may cause the Auditor to pe...,Yes,Yes,The provision explicitly grants and regulates ...
8,2441,Audit Rights,contract_0318,"SHPS shall have the right, upon reasonable pri...",Yes,Yes,The provision explicitly gives SHPS the right ...
9,2725,Audit Rights,contract_0356,"Premier, shall have the right, directly or thr...",Yes,Yes,The provision explicitly gives Premier the rig...


Here I am displaying the previous decisions alongside my own. Next I will highlight the differences.

In [1150]:
disagreements = review_results[
    review_results["my_decision"] != review_results["existing_decision"]
].copy()

print(f"Disagreements: {len(disagreements)}")

display(
    disagreements[
        [
            "review_id",
            "category_name",
            "contract_id",
            "answer_text",
            "my_decision",
            "existing_decision",
            "reasoning"
        ]
    ]
)

Disagreements: 6


,review_id,category_name,contract_id,answer_text,my_decision,existing_decision,reasoning
10,65,Cap On Liability,contract_0012,If a Buyer receives a product that fails to co...,No,Yes,The provision describes the Buyer's remedies f...
13,2351,Cap On Liability,contract_0301,The warranty and remedies set forth in Exhibit...,No,Yes,The provision makes certain warranties and rem...
25,627,License Grant,contract_0092,"Develop, Manufacture, or Commercialize the Pro...",No,Yes,The provision describes activities involving t...
35,359,Revenue/Profit Sharing,contract_0058,HSNS agrees to pay E.piphany an additional ...,No,Yes,The provision requires HSNS to pay E.piphany a...
42,1433,Termination For Convenience,contract_0189,Written notice of intention to withdraw must b...,Unclear,Yes,The provision describes a right to withdraw wi...
49,1967,Uncapped Liability,contract_0249,"Subject to Section 17(c), in no event shall ei...",No,Yes,The provision explicitly establishes a maximum...


Overall the Agreement Rate is 88%, only 6/50 are different.

In [1151]:
agreement_rate = (
    review_results["my_decision"]
    == review_results["existing_decision"]
).mean()

print(f"Agreement rate: {agreement_rate:.1%}")

Agreement rate: 88.0%


# The Report
This is where interpretations and rules are defined

## Recurring Rules about Contract Language

Based on the independent review, the following rules describe the Contract language that should and should not count for each category.

In [1152]:
category_rules = pd.DataFrame([
    {
        "Category": "Governing Law",
        "Language that should count": "Language specifying the law or jurisdiction governing the Agreement",
        "Language that should not count": "References to laws that do not establish the governing law"
    },
    {
        "Category": "Renewal Term",
        "Language that should count": "Automatic renewals, renewal periods, extensions, or conditions for renewal",
        "Language that should not count": "Language describing only the initial term"
    },
    {
        "Category": "Revenue/Profit Sharing",
        "Language that should count": "Revenue, profit, sales, or royalty amounts shared or calculated as a percentage of financial proceeds",
        "Language that should not count": "Ordinary fees, fixed payments, or compensation not tied to shared revenue/profit"
    },
    {
        "Category": "Cap On Liability",
        "Language that should count": "Language establishing a maximum liability amount or limiting types of damages",
        "Language that should not count": "General remedies, warranties, or obligations that do not limit liability"
    },
    {
        "Category": "Uncapped Liability",
        "Language that should count": "Language stating that liability is unlimited or that an existing liability cap does not apply",
        "Language that should not count": "Ordinary liability caps with no applicable exception"
    },
    {
        "Category": "Termination For Convenience",
        "Language that should count": "Termination without requiring breach or another specific triggering event",
        "Language that should not count": "Termination based on breach, insolvency, or another specific cause"
    },
    {
        "Category": "Anti-Assignment",
        "Language that should count": "Restrictions or conditions on assigning/transferring the Agreement, rights, or obligations",
        "Language that should not count": "Provisions unrelated to assignment or transfer"
    },
    {
        "Category": "Audit Rights",
        "Language that should count": "Rights to inspect, review, copy, or audit records, books, or accounts",
        "Language that should not count": "General access provisions that do not provide an inspection or audit right"
    },
    {
        "Category": "License Grant",
        "Language that should count": "Language granting a party permission or rights to use IP, trademarks, materials, or other protected content",
        "Language that should not count": "Restrictions or descriptions of activities that do not themselves grant a right"
    },
    {
        "Category": "Exclusivity",
        "Language that should count": "Exclusive rights or restrictions preventing a party from working with competitors or other parties",
        "Language that should not count": "General commercial restrictions that do not create exclusivity"
    }
])

display(category_rules)

,Category,Language that should count,Language that should not count
0,Governing Law,Language specifying the law or jurisdiction go...,References to laws that do not establish the g...
1,Renewal Term,"Automatic renewals, renewal periods, extension...",Language describing only the initial term
2,Revenue/Profit Sharing,"Revenue, profit, sales, or royalty amounts sha...","Ordinary fees, fixed payments, or compensation..."
3,Cap On Liability,Language establishing a maximum liability amou...,"General remedies, warranties, or obligations t..."
4,Uncapped Liability,Language stating that liability is unlimited o...,Ordinary liability caps with no applicable exc...
5,Termination For Convenience,Termination without requiring breach or anothe...,"Termination based on breach, insolvency, or an..."
6,Anti-Assignment,Restrictions or conditions on assigning/transf...,Provisions unrelated to assignment or transfer
7,Audit Rights,"Rights to inspect, review, copy, or audit reco...",General access provisions that do not provide ...
8,License Grant,Language granting a party permission or rights...,Restrictions or descriptions of activities tha...
9,Exclusivity,Exclusive rights or restrictions preventing a ...,General commercial restrictions that do not cr...


## Known Data Problems and Uncertainties

The independent review identified several category-boundary issues and
examples where additional Contract context would be useful.

### Cap On Liability

Two reviewed examples were classified as No during the independent review
but were classified as Yes in the existing annotations. These provisions
described remedies or warranty limitations but did not clearly establish a
maximum liability amount or limitation on damages.

This suggests that the distinction between limiting remedies or warranties
and limiting liability should be made more explicit.

### License Grant

One example was classified as No because it described activities involving a
licensed field but did not itself grant permission or rights to use
intellectual property.

This indicates that references to a license or licensed field should not
automatically be treated as a license grant.

### Revenue/Profit Sharing

One example was classified as No because it required a fixed per-email
payment related to a deal. The provision described a payment obligation but
did not clearly involve sharing revenue or profit.

This creates a boundary between revenue/profit sharing and ordinary fees,
commissions, or other compensation.

### Termination For Convenience

One example was marked Unclear because it described a right to withdraw with
advance notice, but the excerpt did not provide enough context to determine
whether the withdrawal constituted termination for convenience.

Additional surrounding Contract language would be needed to resolve this
example.

### Uncapped Liability

One example was classified as No because the excerpt explicitly established
a maximum liability amount. However, it referenced an exception in another
section that was not included in the excerpt.

This demonstrates that liability provisions may require surrounding sections
to determine whether an exception creates uncapped liability.

## Relationships Between Categories

Several categories are closely related and can be difficult to distinguish
when Contract language addresses multiple concepts at once.



**Cap On Liability and Uncapped Liability** are directly related. A Contract
may establish a general liability cap while creating exceptions for certain
claims. The cap establishes the general limitation, while the exception may
create uncapped liability for the specified claims.

**License Grant and Exclusivity** can also appear together. A Contract may
grant a party a license while separately making that license exclusive.
However, the existence of a license does not by itself establish exclusivity.

**Renewal Term and Termination For Convenience** both describe the duration
of a Contract but address different events. Renewal language describes how
the Contract continues after its term, while termination-for-convenience
language describes how a party can end the Contract before its scheduled
expiration.

There is also the case where related context is not included. Some excerpts do not contain enough surrounding Contract context to resolve these relationships. Referenced sections, definitions, exhibits, or exceptions may be necessary to determine the correct category.

## Independence From Official-Test Information


The category selection and independent review were performed using the training-approved data and the selected category definitions.

No official-test statistics, official-test source-text excerpts, official-test labels, or official-test results were used to select the categories or make the independent review decisions.

The review was performed on a sample of Contract evidence from the training split before I compared my decisons with thepreexisting annotations.

## Reviewer and Evidence Info

 **Independent reviewer:** Quinn Hinde-Schuster

**Review purpose:** Independent review of annotation clarity and category
boundaries.

**Evidence reviewed:** 50 Contract evidence spans, consisting of 5 randomly
selected examples from each of the 10 final categories.

**Sampling method:** Random sampling within each category using
`random_state=42`.

**Categories reviewed:**
- Governing Law
- Renewal Term
- Revenue/Profit Sharing
- Cap On Liability
- Uncapped Liability
- Termination For Convenience
- Anti-Assignment
- Audit Rights
- License Grant
- Exclusivity

**Data split:** Training portion of the frozen train/validation split.

**Review outcome:** 44 of 50 examples agreed with the preexisting decision,
for an 88% agreement rate. Six examples were disagreements or uncertainties.

**Comparison:** The existing `is_impossible` annotations were revealed only
after the independent decisions were recorded, allowing the for
initial decisions to be made independently.

## Exact Versions/IDs of the evidence used

In [1153]:
review_audit_trail = review_results[
    [
        "review_id",
        "category_name",
        "contract_id",
        "span_id",
        "annotation_set_id",
        "my_decision",
        "existing_decision"
    ]
].copy()

display(review_audit_trail)

,review_id,category_name,contract_id,span_id,annotation_set_id,my_decision,existing_decision
0,270,Anti-Assignment,contract_0042,contract_0042__anti_assignment__span_000,contract_0042__anti_assignment,Yes,Yes
1,337,Anti-Assignment,contract_0056,contract_0056__anti_assignment__span_000,contract_0056__anti_assignment,Yes,Yes
2,507,Anti-Assignment,contract_0073,contract_0073__anti_assignment__span_001,contract_0073__anti_assignment,Yes,Yes
3,1676,Anti-Assignment,contract_0214,contract_0214__anti_assignment__span_006,contract_0214__anti_assignment,Yes,Yes
4,1948,Anti-Assignment,contract_0249,contract_0249__anti_assignment__span_001,contract_0249__anti_assignment,Yes,Yes
5,941,Audit Rights,contract_0137,contract_0137__audit_rights__span_001,contract_0137__audit_rights,Yes,Yes
6,1122,Audit Rights,contract_0160,contract_0160__audit_rights__span_001,contract_0160__audit_rights,Yes,Yes
7,1296,Audit Rights,contract_0177,contract_0177__audit_rights__span_001,contract_0177__audit_rights,Yes,Yes
8,2441,Audit Rights,contract_0318,contract_0318__audit_rights__span_000,contract_0318__audit_rights,Yes,Yes
9,2725,Audit Rights,contract_0356,contract_0356__audit_rights__span_002,contract_0356__audit_rights,Yes,Yes
